<a href="https://colab.research.google.com/github/prakash8806/gen_ai_ml/blob/main/Task_Manager_with_User_Authentication.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import csv
import hashlib
from datetime import datetime

USERS_FILE = "users.txt"


def hash_password(password):
    return hashlib.sha256(password.encode()).hexdigest()


def register():
    print("\n--- User Registration ---")
    username = input("Enter username: ")

    if os.path.exists(USERS_FILE):
        with open(USERS_FILE, "r") as file:
            for line in file:
                if line.split(",")[0] == username:
                    print("❌ Username already exists!")
                    return None

    password = input("Enter password: ")
    hashed = hash_password(password)

    with open(USERS_FILE, "a") as file:
        file.write(f"{username},{hashed}\n")

    print("✅ Registration successful!")
    return username


def login():
    print("\n--- User Login ---")
    username = input("Enter username: ")
    password = input("Enter password: ")
    hashed = hash_password(password)

    if not os.path.exists(USERS_FILE):
        print("❌ No users registered yet.")
        return None

    with open(USERS_FILE, "r") as file:
        for line in file:
            stored_user, stored_pass = line.strip().split(",")
            if stored_user == username and stored_pass == hashed:
                print("✅ Login successful!")
                return username

    print("❌ Invalid credentials!")
    return None


def task_file(user):
    return f"{user}_tasks.txt"


def add_task(user):
    desc = input("Enter task description: ")
    task_id = int(datetime.now().timestamp())
    status = "Pending"

    with open(task_file(user), "a") as file:
        file.write(f"{task_id},{desc},{status}\n")

    print("✅ Task added successfully!")


def view_tasks(user):
    print("\n--- Your Tasks ---")
    if not os.path.exists(task_file(user)):
        print("No tasks found.")
        return

    with open(task_file(user), "r") as file:
        for line in file:
            task_id, desc, status = line.strip().split(",")
            print(f"ID: {task_id} | Task: {desc} | Status: {status}")


def complete_task(user):
    view_tasks(user)
    task_id = input("Enter Task ID to mark completed: ")

    updated = []
    found = False

    with open(task_file(user), "r") as file:
        for line in file:
            tid, desc, status = line.strip().split(",")
            if tid == task_id:
                status = "Completed"
                found = True
            updated.append(f"{tid},{desc},{status}\n")

    if found:
        with open(task_file(user), "w") as file:
            file.writelines(updated)
        print("✅ Task marked as completed!")
    else:
        print("❌ Task not found!")


def delete_task(user):
    view_tasks(user)
    task_id = input("Enter Task ID to delete: ")

    updated = []
    found = False

    with open(task_file(user), "r") as file:
        for line in file:
            tid, desc, status = line.strip().split(",")
            if tid != task_id:
                updated.append(line)
            else:
                found = True

    if found:
        with open(task_file(user), "w") as file:
            file.writelines(updated)
        print("✅ Task deleted!")
    else:
        print("❌ Task not found!")


def expense_file(user):
    return f"{user}_expenses.csv"


def budget_file(user):
    return f"{user}_budget.txt"


def set_budget(user):
    amount = float(input("Enter monthly budget: $"))
    with open(budget_file(user), "w") as file:
        file.write(str(amount))
    print("✅ Budget saved!")


def get_budget(user):
    if not os.path.exists(budget_file(user)):
        return 0
    with open(budget_file(user), "r") as file:
        return float(file.read())


def add_expense(user):
    date = input("Enter date (YYYY-MM-DD): ")
    category = input("Enter category: ")
    amount = float(input("Enter amount: $"))
    desc = input("Enter description: ")

    file_exists = os.path.exists(expense_file(user))

    with open(expense_file(user), "a", newline="") as file:
        writer = csv.writer(file)
        if not file_exists:
            writer.writerow(["Date", "Category", "Amount", "Description"])
        writer.writerow([date, category, amount, desc])

    print("✅ Expense added!")


def view_expenses(user):
    print("\n--- Your Expenses ---")
    if not os.path.exists(expense_file(user)):
        print("No expenses found.")
        return

    with open(expense_file(user), "r") as file:
        reader = csv.reader(file)
        for row in reader:
            print(row)


def track_budget(user):
    budget = get_budget(user)
    if budget == 0:
        print("❌ No budget set.")
        return

    total = 0
    if os.path.exists(expense_file(user)):
        with open(expense_file(user), "r") as file:
            reader = csv.DictReader(file)
            for row in reader:
                total += float(row["Amount"])

    print(f"\n💰 Budget: ${budget}")
    print(f"💸 Total Expenses: ${total}")

    if total > budget:
        print("⚠️ You have exceeded your budget!")
    else:
        print(f"✅ Remaining Balance: ${budget - total}")


def task_menu(user):
    while True:
        print("\n--- Task Manager ---")
        print("1. Add Task")
        print("2. View Tasks")
        print("3. Mark Task as Completed")
        print("4. Delete Task")
        print("5. Logout")

        choice = input("Choose option: ")

        if choice == "1":
            add_task(user)
        elif choice == "2":
            view_tasks(user)
        elif choice == "3":
            complete_task(user)
        elif choice == "4":
            delete_task(user)
        elif choice == "5":
            break
        else:
            print("❌ Invalid choice!")


def budget_menu(user):
    while True:
        print("\n--- Budget & Expense Manager ---")
        print("1. Set Budget")
        print("2. Add Expense")
        print("3. View Expenses")
        print("4. Track Budget")
        print("5. Exit")

        choice = input("Choose option: ")

        if choice == "1":
            set_budget(user)
        elif choice == "2":
            add_expense(user)
        elif choice == "3":
            view_expenses(user)
        elif choice == "4":
            track_budget(user)
        elif choice == "5":
            break
        else:
            print("❌ Invalid choice!")


def main():
    while True:
        print("\n--- Welcome ---")
        print("1. Register")
        print("2. Login")
        print("3. Exit")

        choice = input("Choose option: ")

        if choice == "1":
            register()
        elif choice == "2":
            user = login()
            if user:
                task_menu(user)
                budget_menu(user)
        elif choice == "3":
            print("Goodbye 👋")
            break
        else:
            print("❌ Invalid choice!")

main()



--- Welcome ---
1. Register
2. Login
3. Exit
Choose option: 1

--- User Registration ---
Enter username: prakash
Enter password: prakash
✅ Registration successful!

--- Welcome ---
1. Register
2. Login
3. Exit
Choose option: 2

--- User Login ---
Enter username: prakash
Enter password: prakash
✅ Login successful!

--- Task Manager ---
1. Add Task
2. View Tasks
3. Mark Task as Completed
4. Delete Task
5. Logout
Choose option: 1
Enter task description: food
✅ Task added successfully!

--- Task Manager ---
1. Add Task
2. View Tasks
3. Mark Task as Completed
4. Delete Task
5. Logout
Choose option: 2

--- Your Tasks ---
ID: 1770007807 | Task: food | Status: Pending

--- Task Manager ---
1. Add Task
2. View Tasks
3. Mark Task as Completed
4. Delete Task
5. Logout
Choose option: 3

--- Your Tasks ---
ID: 1770007807 | Task: food | Status: Pending
Enter Task ID to mark completed: 1770007807
✅ Task marked as completed!

--- Task Manager ---
1. Add Task
2. View Tasks
3. Mark Task as Completed
4.